In [1]:
!pip install transformers datasets accelerate evaluate

In [2]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from torch.utils.data import Dataset

In [3]:
news_df = pd.read_csv("../data/processed/final_news.csv")

In [4]:
news_df.head()

,title,text,subject,date,label,text_length,content
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,"A school librarian in Cambridge, Massachusetts...",left-news,"Sep 28, 2017",0,10062,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...
1,Kenya opposition leader calls for calm in slum...,NAIROBI (Reuters) - Kenyan opposition leader R...,worldnews,"October 29, 2017",1,2808,Kenya opposition leader calls for calm in slum...
2,Egypt rejects U.S. decision to move its embass...,CAIRO (Reuters) - Egypt rejected the U.S. deci...,worldnews,"December 6, 2017",1,232,Egypt rejects U.S. decision to move its embass...
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,After a recent speech given by Minister Louis ...,left-news,"May 8, 2015",0,610,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,Tensions ran high outside of a campaign rally ...,News,"March 11, 2016",0,2658,Trump Rally Nearly Turns Into A Full-Blown Rac...


In [5]:
print(news_df.shape)

(44058, 7)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    news_df["content"],
    news_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=news_df["label"]
)

In [7]:
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 35246
Testing Samples: 8812


In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [9]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()

train_labels = y_train.tolist()
test_labels = y_test.tolist()

In [10]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [11]:
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [12]:
class NewsDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [13]:
train_dataset = NewsDataset(train_encodings, train_labels)

test_dataset = NewsDataset(test_encodings, test_labels)

In [14]:
print(len(train_dataset))
print(len(test_dataset))

35246
8812


In [15]:
train_dataset[0]

{'input_ids': tensor([  101,  1520,  2025,  1011, 16939,  1521,  8398,  5470,  3632,  2006,
          2847,  1011,  2146, 10474,  2743,  2102,  2055,  1520,  9152,  5620,
          1521,  1998,  1520,  8112, 19093,  1521,  1006,  1056, 28394,  3215,
          1007,  2065,  2017,  2412,  2342,  2000,  2113,  3599,  2129,  4487,
         11365,  3512,  6221,  8398,  2038,  2042,  1010,  2065,  2017,  2412,
          2342,  1037,  3819,  2742,  1010,  2017,  2123,  1056,  2342,  2000,
          2298,  2172,  2582,  2084,  6221,  8398,  1055, 10474, 17060,  1012,
          2096,  2009,  2003,  1037,  8292,  4757, 23270,  1997, 16939, 13044,
          2006,  1037,  2204,  2154,  1010,  2823,  2010,  4599,  3233,  2041,
          1998,  2028,  1997,  2068,  4240,  2004,  1037,  3819, 14764,  1997,
          2054,  1010,  3599,  2057,  2024,  2157,  2075,  2114,  1012,  2006,
          4465,  1010,  8398, 23678,  2098,  2055,  2010,  8599,  1999,  1037,
          7143,  3947,  2000, 15886,  2

In [16]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [18]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
print(next(model.parameters()).device)

cuda:0


In [20]:
import transformers
print(transformers.__version__)

5.14.1


In [21]:
training_args = TrainingArguments(
    output_dir="../results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    fp16=True
)

In [22]:
print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [24]:
print(trainer.args.per_device_train_batch_size)
print(trainer.args.num_train_epochs)

4
2


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

cuda


In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000030,0.015297
2,0.000002,0.000039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17624, training_loss=0.006290808404595864, metrics={'train_runtime': 34561.5907, 'train_samples_per_second': 2.04, 'train_steps_per_second': 0.51, 'total_flos': 9273612257218560.0, 'train_loss': 0.006290808404595864, 'epoch': 2.0})

In [27]:
model.save_pretrained("../models/fake_news_bert")
tokenizer.save_pretrained("../models/fake_news_bert")

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!
